# Invasive species: bamboo encroachment risk

### Step 0: import required packages and set up base path

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib
print(matplotlib.rcParams['font.family'])
matplotlib.rcParams['font.family'] = 'Times New Roman'

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

### Step 1: load the layers for land use and categorise them

In [ ]:
land_use = base_path / "2013_landuse_LandCover.shp"

In [ ]:
terrestrial_landcover = gpd.read_file(land_use)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print(terrestrial_landcover.crs)

In [ ]:
# Re-write these, some of the mixed classes are wrong - don't reflect bamboo / agri
# landuse_category_mapping = {
#     'Bare Rock': 'Bare Rock',
#     'Fields: Herbaceous crops, fallow, cultivated vegetables': 'Agriculture',
#     'Fields: Pasture,Human disturbed, grassland': 'Agriculture',
#     'Herbaceous Wetland': 'Freshwater wetland',
#     'Mangrove Forest': 'Mangrove',
#     'Fields: Bare Land': 'Agriculture',
#     'Open dry forest - Short': 'Open dry forest',
#     'Open dry forest - Tall (Woodland/Savanna)': 'Open dry forest',
#     'Plantation: Tree crops, shrub crops, sugar cane, banana': 'Plantation',
#     'Quarry': 'Bauxite extraction / quarry',
#     'Water Body': 'Water body',
#     'Buildings and other infrastructures': 'Buildings and other infrastructure',
#     'Fields and Secondary Forest': 'Mixed land use: forests with bamboo or agriculture/plantation',
#     'Bamboo and Fields': 'Mixed land use: agriculture and bamboo',
#     'Bauxite Extraction': 'Bauxite extraction / quarry',
#     'Disturbed broadleaved forest (Secondary Forest)': 'Forest',
#     'Fields  and Bamboo': 'Mixed land use: agriculture and bamboo',
#     'Bamboo and Secondary Forest': 'Mixed land use: agriculture and bamboo',
#     'Hardwood Plantation: Euculytus': 'Plantation',
#     'Hardwood Plantation: Mixed': 'Plantation',
#     'Swamp Forest': 'Swamp forest',
#     'Fields or Secondary Forest/Pine Plantation': 'Mixed land use: forests with bamboo or agriculture/plantation',
#     'Hardwood Plantation: Mahoe': 'Plantation',
#     'Hardwood Plantation: Mahogany': 'Plantation',
#     'Bamboo': 'Bamboo',
#     'Closed broadleaved forest (Primary Forest)': 'Forest',
#     'Secondary Forest': 'Forest'
#     # Add other mappings as needed
# }


# Re-write these, some of the mixed classes are wrong - don't reflect bamboo / agri
landuse_category_mapping = {
    'Bare Rock': 'Bare Rock',
    'Fields: Herbaceous crops, fallow, cultivated vegetables': 'Agriculture',
    'Fields: Pasture,Human disturbed, grassland': 'Agriculture',
    'Herbaceous Wetland': 'Freshwater wetland',
    'Mangrove Forest': 'Mangrove',
    'Fields: Bare Land': 'Agriculture',
    'Open dry forest - Short': 'Open dry forest: short', # this one I have changed too, see?
    'Open dry forest - Tall (Woodland/Savanna)': 'Open dry forest: tall', # this one I have changed too, see?
    'Plantation: Tree crops, shrub crops, sugar cane, banana': 'Plantation',
    'Quarry': 'Bauxite extraction / quarry',
    'Water Body': 'Water body',
    'Buildings and other infrastructures': 'Buildings and other infrastructure',
    'Fields and Secondary Forest': 'Mixed land use: agriculture with secondary forest', # this one I have changed too, see?
    'Bamboo and Fields': 'Mixed land use: agriculture and bamboo',
    'Bauxite Extraction': 'Bauxite extraction / quarry',
    'Disturbed broadleaved forest (Secondary Forest)': 'Disturbed broadleaved forest', # this one I have changed, see?
    'Fields  and Bamboo': 'Mixed land use: agriculture and bamboo',
    'Bamboo and Secondary Forest': 'Mixed land use: bamboo and forest', # this one I have changed too, see?
    'Hardwood Plantation: Euculytus': 'Plantation',
    'Hardwood Plantation: Mixed': 'Plantation',
    'Swamp Forest': 'Swamp forest',
    'Fields or Secondary Forest/Pine Plantation': 'Mixed land use: agriculture with forest or pine plantation', # this one I have changed too, see?
    'Hardwood Plantation: Mahoe': 'Plantation',
    'Hardwood Plantation: Mahogany': 'Plantation',
    'Bamboo': 'Bamboo',
    'Closed broadleaved forest (Primary Forest)': 'Closed broadleaved forest', # this one I have changed too, see?
    'Secondary Forest': 'Secondary forest' # this one I have changed too, see?
    # Add other mappings as needed
}


# Map the categories
terrestrial_landcover['Classify'] = terrestrial_landcover['Classify'].replace(landuse_category_mapping)

In [ ]:
categories = terrestrial_landcover['Classify'].unique()
# Display the categories
print("Land Use Categories:")
for category in categories:
    print("-", category)

In [ ]:
# Step 1: Define the bamboo-related categories
bamboo_categories = [
    "Bamboo",  # Assuming this category explicitly represents bamboo
    "Mixed land use: agriculture and bamboo",  # Mixed categories that include bamboo
    "Mixed land use: forests with bamboo or agriculture/plantation",  # Other mixed bamboo categories
    "Fields and Bamboo",  # Alternate naming
    "Bamboo and Secondary Forest"  # Bamboo with forests
]


In [ ]:
# Step 2: Filter the land use GeoDataFrame to include only bamboo-related categories
bamboo_landcover = terrestrial_landcover[terrestrial_landcover['Classify'].isin(bamboo_categories)]


In [ ]:
# Step 3: Display the filtered categories
display("Filtered Bamboo Landcover:")
display(bamboo_landcover.head())


In [ ]:
# Step 4: Plot the bamboo landcover for a quick visualization
bamboo_landcover.plot(
    figsize=(10, 8),
    color="beige",  # Bamboo-specific color
    edgecolor="black",
    linewidth=0.5
)
plt.title("Bamboo-Related Landcover in Jamaica", fontsize=16)
plt.show()


In [ ]:
# Paths to protected areas shapefiles
forest_reserves_path = base_path / "protected_landcover/Forest_reserves.shp"
protected_areas_path = base_path / "protected_landcover/Protected_areas.shp"

# Load the protected areas shapefiles
forest_reserves = gpd.read_file(forest_reserves_path)
protected_areas = gpd.read_file(protected_areas_path)

# Reproject both layers to the same CRS as the landcover and bauxite reserves
forest_reserves = forest_reserves.to_crs(jamaica_metric_grid_crs)
protected_areas = protected_areas.to_crs(jamaica_metric_grid_crs)

# Combine the protected areas using a union operation
combined_protected_layers = gpd.overlay(forest_reserves, protected_areas, how='union')

# Inspect the combined protected layers to verify
display(combined_protected_layers.head())
display(combined_protected_layers.crs)

In [ ]:
# Step 2: Intersect Bamboo Landcover with Protected Areas
protected_bamboo = gpd.overlay(bamboo_landcover, combined_protected_layers, how='intersection')

In [ ]:
protected_bamboo.plot()

In [ ]:
# Step 3: Calculate Overlap Area
protected_bamboo['area_m2'] = protected_bamboo.geometry.area  # Area in square meters

# Group by land use type to summarize the protected areas
protected_bamboo_summary = (
    protected_bamboo.groupby('Classify')['area_m2']
    .sum()
    .reset_index()
)
# Add area in km² and percentage
protected_bamboo_summary['area_km2'] = protected_bamboo_summary['area_m2'] / 1e6
protected_bamboo_summary['percentage'] = (
    protected_bamboo_summary['area_m2'] / bamboo_landcover.geometry.area.sum()
) * 100


# Rename columns for clarity
protected_bamboo_summary.columns = ['Land Use Type', 'Protected Area (m²)', 'Protected Area (km²)', 'Percentage (%)']

# Step 4: Display the Results
print("Protected Bamboo-Related Landcover Summary:")
display(protected_bamboo_summary)

# Step 5: Visualize the Overlap
fig, ax = plt.subplots(figsize=(12, 10))

# Plot bamboo landcover
bamboo_landcover.plot(
    ax=ax,
    color='green',
    edgecolor='black',
    linewidth=0.3,
    label="Bamboo Landcover"
)

# Plot protected bamboo areas
protected_bamboo.plot(
    ax=ax,
    color='lightblue',
    edgecolor='blue',
    linewidth=0.3,
    label="Protected Bamboo"
)

# Plot Jamaica boundary for reference
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    label="Jamaica Boundary"
)

# Add title and legend
plt.title("Protected Areas Overlapping Bamboo-Related Landcover", fontsize=16)
plt.legend(loc='upper left')
plt.show()

In [ ]:
# Set colours

custom_colors = {
    'Bare Rock': '#A9A9A9',  # Dark Gray (rocky terrain)
    'Agriculture': '#8B4513',  # Dark Brown
    'Freshwater wetland': '#4169E1', #Royal blue
    'Mangrove': '#008080', # Teal Blue
    'Open dry forest': '#A4C639', # Light green 90EE90   
    'Plantation': '#F5DEB3', #  yellow
    'Bauxite extraction / quarry': '#B22222', #Iron Oxide Red
    'Water body': '#4682B4',  # Steel Blue
    'Buildings and other infrastructure': '#000000',  # Black
    'Mixed land use: forests with bamboo or agriculture/plantation': '#32CD32',  # lime Green
    'Mixed land use: agriculture and bamboo': '#D2B48C',  #light brown
    'Swamp forest': '#6B8E23',  # Olive Drab
    'Bamboo': '#F4A460',  # Sandy Brown
    'Forest': '#006400',  # Dark Green
}

# Categories in the data
categories = terrestrial_landcover['Classify'].unique()

# Create the color mapping for all categories
category_colors = {cat: custom_colors.get(cat, '#808080') for cat in categories}  # Default to grey if not specified

# Assign colors to the land use GeoDataFrame
terrestrial_landcover['color'] = terrestrial_landcover['Classify'].map(category_colors)


In [ ]:
# Step 1: Intersect Protected Areas with Land Use
protected_landuse = gpd.overlay(terrestrial_landcover, combined_protected_layers, how='intersection')

In [ ]:
# Step 2: Highlight Bamboo-Related Classes in Protected Areas
bamboo_categories = [
    "Bamboo",  # Pure bamboo category
    "Mixed land use: agriculture and bamboo",  # Mixed use with bamboo
    "Mixed land use: forests with bamboo or agriculture/plantation",  # Mixed forest with bamboo
    "Fields and Bamboo",  # Fields and bamboo
    "Bamboo and Secondary Forest"  # Secondary forest mixed with bamboo
]

# Filter to get bamboo-related classes in protected areas
protected_bamboo = protected_landuse[protected_landuse['Classify'].isin(bamboo_categories)]

# Assign a distinct color (blue) for bamboo-related classes within protected areas
protected_bamboo['color'] = 'blue'

# Step 3: Assign Normal Colors for Other Classes
protected_landuse['color'] = protected_landuse['Classify'].map(category_colors)

# Ensure bamboo-related classes overwrite other colors
protected_landuse.loc[protected_landuse['Classify'].isin(bamboo_categories), 'color'] = 'blue'

# Step 4: Plot the Map
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot all protected land use
protected_landuse.plot(
    ax=ax,
    color=protected_landuse['color'],
    edgecolor='black',
    linewidth=0.1,
    label="Protected Land Use"
)

# Highlight bamboo-related classes in blue
protected_bamboo.plot(
    ax=ax,
    color='blue',
    edgecolor='black',
    linewidth=0.1,
    label="Bamboo-Related Classes in Protected Areas"
)

# Plot Jamaica boundary for reference
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    label="Jamaica Boundary"
)

# Prepare legend handles
legend_handles = [
    mpatches.Patch(color=color, label=f"{land_use}")
    for land_use, color in category_colors.items()
    if land_use not in bamboo_categories
]

# Add a legend entry for bamboo-related classes
legend_handles.append(mpatches.Patch(color='blue', label="Bamboo-Related Classes"))

# Add the legend
legend = ax.legend(
    handles=legend_handles,
    title="Land Use in Protected Areas",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

def add_scale_bar(ax, length_km=20, location=(0.9, 0.8), linewidth=2, tick_height=0.01, label_offset=0.02, km_offset=0.03):
    """
    Add a scale bar to a plot, with "20" under the last tick mark and "km" positioned slightly to the right of the scale bar.
    """
    x, y = location  # Adjusted location (higher y value to move the scale bar upward)
    bar_half_length = 0.05  # Half the scale bar length in axes fraction

    # Draw the scale bar
    ax.plot(
        [x - bar_half_length, x + bar_half_length], [y, y],  # Scale bar endpoints
        transform=ax.transAxes, color='black', linewidth=linewidth
    )

    # Draw perpendicular tick marks
    tick_positions = [x - bar_half_length, x, x + bar_half_length]
    for pos in tick_positions:
        ax.plot(
            [pos, pos], [y - tick_height / 2, y + tick_height / 2],  # Vertical line for ticks
            transform=ax.transAxes, color='black', linewidth=linewidth
        )

    # Add numeric labels below the tick marks
    ax.text(
        x - bar_half_length, y - tick_height - label_offset, "0", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x, y - tick_height - label_offset, f"{length_km // 2}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x + bar_half_length, y - tick_height - label_offset, f"{length_km}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )

    # Add "km" label slightly to the right of the scale bar
    ax.text(
        x + bar_half_length + km_offset, y, "km", transform=ax.transAxes, 
        ha='left', va='center', fontsize=12
    )

def add_north_arrow(ax, location=(0.9, 0.8), size=0.05, fontsize=12, label_offset=0.03):
    """
    Add a north arrow to the plot, with "N" positioned slightly above the arrow.
    """
    x, y = location

    # Draw the arrow
    ax.annotate(
        '', xy=(x, y + size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', headwidth=10, headlength=15, width=5)
    )

    # Add the "N" label slightly above the arrow
    ax.text(
        x, y + size + label_offset, "N", transform=ax.transAxes,
        fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black"
    )



# Add north arrow and scale bar
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add title
plt.title(
    "Land Use in Protected Areas with Bamboo-Related Classes Highlighted",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
# Assign a single color for general protected land use
protected_landuse['color'] = 'lightgrey'

# Assign a distinct color (blue) for bamboo-related classes within protected areas
protected_landuse.loc[protected_landuse['Classify'].isin(bamboo_categories), 'color'] = 'blue'

# Step 3: Plot the Map
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot all protected land use in light grey
protected_landuse.plot(
    ax=ax,
    color=protected_landuse['color'],
    edgecolor='black',
    linewidth=0.1,
    label="Protected Land Use"
)

# Highlight bamboo-related classes in blue
protected_bamboo.plot(
    ax=ax,
    color='blue',
    edgecolor='black',
    linewidth=0.1,
    label="Bamboo-Related Classes in Protected Areas"
)

# Plot Jamaica boundary for reference
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    label="Jamaica Boundary"
)

# Prepare legend handles
legend_handles = [
    mpatches.Patch(color='lightgrey', label="Protected Land Use"),
    mpatches.Patch(color='blue', label="Bamboo-Related Classes")
]

# Add the legend
legend = ax.legend(
    handles=legend_handles,
    title="Land Use in Protected Areas",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=2,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

def add_scale_bar(ax, length_km=20, location=(0.9, 0.8), linewidth=2, tick_height=0.01, label_offset=0.02, km_offset=0.03):
    """
    Add a scale bar to a plot, with "20" under the last tick mark and "km" positioned slightly to the right of the scale bar.
    """
    x, y = location  # Adjusted location (higher y value to move the scale bar upward)
    bar_half_length = 0.05  # Half the scale bar length in axes fraction

    # Draw the scale bar
    ax.plot(
        [x - bar_half_length, x + bar_half_length], [y, y],  # Scale bar endpoints
        transform=ax.transAxes, color='black', linewidth=linewidth
    )

    # Draw perpendicular tick marks
    tick_positions = [x - bar_half_length, x, x + bar_half_length]
    for pos in tick_positions:
        ax.plot(
            [pos, pos], [y - tick_height / 2, y + tick_height / 2],  # Vertical line for ticks
            transform=ax.transAxes, color='black', linewidth=linewidth
        )

    # Add numeric labels below the tick marks
    ax.text(
        x - bar_half_length, y - tick_height - label_offset, "0", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x, y - tick_height - label_offset, f"{length_km // 2}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x + bar_half_length, y - tick_height - label_offset, f"{length_km}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )

    # Add "km" label slightly to the right of the scale bar
    ax.text(
        x + bar_half_length + km_offset, y, "km", transform=ax.transAxes, 
        ha='left', va='center', fontsize=12
    )

def add_north_arrow(ax, location=(0.9, 0.8), size=0.05, fontsize=12, label_offset=0.03):
    """
    Add a north arrow to the plot, with "N" positioned slightly above the arrow.
    """
    x, y = location

    # Draw the arrow
    ax.annotate(
        '', xy=(x, y + size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', headwidth=10, headlength=15, width=5)
    )

    # Add the "N" label slightly above the arrow
    ax.text(
        x, y + size + label_offset, "N", transform=ax.transAxes,
        fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black"
    )
    
# Add north arrow and scale bar
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add title
plt.title(
    "Protected Land Use and Bamboo-Related Classes in Protected Areas",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
# Bamboo in relation to forests

In [ ]:
forest_categories = [
    "Forest",
    "Closed broadleaved forest (Primary Forest)",
    "Disturbed broadleaved forest (Secondary Forest)",
    "Swamp Forest"
]

# Step 2: Filter Bamboo and Forest Classes
bamboo_layer = terrestrial_landcover[terrestrial_landcover['Classify'].isin(bamboo_categories)]
forest_layer = terrestrial_landcover[terrestrial_landcover['Classify'].isin(forest_categories)]


In [ ]:
# Step 3: Find Intersection/Overlap Between Bamboo and Forest Classes
bamboo_forest_intersection = gpd.overlay(bamboo_layer, forest_layer, how='intersection')

In [ ]:
# Step 4: Assign Colors
bamboo_layer['color'] = 'blue'
forest_layer['color'] = 'green'
bamboo_forest_intersection['color'] = 'purple'

In [ ]:
# Step 5: Plot the Map
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot Forest Areas in Green
forest_layer.plot(
    ax=ax,
    color=forest_layer['color'],
    edgecolor='black',
    linewidth=0.1,
    label="Forest Classes"
)

# Plot Bamboo Areas in Blue
bamboo_layer.plot(
    ax=ax,
    color=bamboo_layer['color'],
    edgecolor='black',
    linewidth=0.1,
    label="Bamboo Classes"
)

# Plot Bamboo-Forest Overlaps in Purple
bamboo_forest_intersection.plot(
    ax=ax,
    color=bamboo_forest_intersection['color'],
    edgecolor='black',
    linewidth=0.1,
    label="Bamboo-Forest Overlap"
)

# Plot Jamaica boundary for reference
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    label="Jamaica Boundary"
)

# Step 6: Add Legend
legend_handles = [
    mpatches.Patch(color='green', label="Forest Classes"),
    mpatches.Patch(color='blue', label="Bamboo Classes"),
    mpatches.Patch(color='purple', label="Bamboo-Forest Overlap")
]

legend = ax.legend(
    handles=legend_handles,
    title="Forests and Bamboo Classes",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

def add_scale_bar(ax, length_km=20, location=(0.9, 0.8), linewidth=2, tick_height=0.01, label_offset=0.02, km_offset=0.03):
    """
    Add a scale bar to a plot, with "20" under the last tick mark and "km" positioned slightly to the right of the scale bar.
    """
    x, y = location  # Adjusted location (higher y value to move the scale bar upward)
    bar_half_length = 0.05  # Half the scale bar length in axes fraction

    # Draw the scale bar
    ax.plot(
        [x - bar_half_length, x + bar_half_length], [y, y],  # Scale bar endpoints
        transform=ax.transAxes, color='black', linewidth=linewidth
    )

    # Draw perpendicular tick marks
    tick_positions = [x - bar_half_length, x, x + bar_half_length]
    for pos in tick_positions:
        ax.plot(
            [pos, pos], [y - tick_height / 2, y + tick_height / 2],  # Vertical line for ticks
            transform=ax.transAxes, color='black', linewidth=linewidth
        )

    # Add numeric labels below the tick marks
    ax.text(
        x - bar_half_length, y - tick_height - label_offset, "0", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x, y - tick_height - label_offset, f"{length_km // 2}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x + bar_half_length, y - tick_height - label_offset, f"{length_km}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )

    # Add "km" label slightly to the right of the scale bar
    ax.text(
        x + bar_half_length + km_offset, y, "km", transform=ax.transAxes, 
        ha='left', va='center', fontsize=12
    )

def add_north_arrow(ax, location=(0.9, 0.8), size=0.05, fontsize=12, label_offset=0.03):
    """
    Add a north arrow to the plot, with "N" positioned slightly above the arrow.
    """
    x, y = location

    # Draw the arrow
    ax.annotate(
        '', xy=(x, y + size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', headwidth=10, headlength=15, width=5)
    )

    # Add the "N" label slightly above the arrow
    ax.text(
        x, y + size + label_offset, "N", transform=ax.transAxes,
        fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black"
    )
    
# Add north arrow and scale bar
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Step 7: Add Title
plt.title(
    "Forests and Bamboo Classes in Jamaica",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
# Spatial join to find forests within 500m of bamboo
forest_near_bamboo = gpd.sjoin_nearest(
    forest_layer,
    bamboo_layer,
    max_distance=500,
    how='inner'
)

# Calculate the total area of forests near bamboo
forest_near_bamboo['area_m2'] = forest_near_bamboo.geometry.area
total_area_near_bamboo_km2 = forest_near_bamboo['area_m2'].sum() / 1e6
print(f"Total forest area near bamboo: {total_area_near_bamboo_km2:.2f} km²")

In [ ]:
forest_near_bamboo.plot()
forest_layer.plot()

In [ ]:
import rasterio
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt

# Define raster parameters
resolution = 10  # Adjust as needed
bounds = forest_layer.total_bounds
width = int((bounds[2] - bounds[0]) / resolution)
height = int((bounds[3] - bounds[1]) / resolution)

# Rasterize bamboo layer
bamboo_raster = rasterize(
    [(geom, 1) for geom in bamboo_layer.geometry],
    out_shape=(height, width),
    transform=rasterio.transform.from_bounds(*bounds, width, height)
)

# Compute proximity (distance transform)
bamboo_proximity = distance_transform_edt(bamboo_raster == 0) * resolution

# Threshold for 500m proximity
forest_within_proximity = (bamboo_proximity <= 500).astype(int)

# Calculate forest area at risk
forest_raster = rasterize(
    [(geom, 1) for geom in forest_layer.geometry],
    out_shape=(height, width),
    transform=rasterio.transform.from_bounds(*bounds, width, height)
)

forest_at_risk = forest_raster * forest_within_proximity
total_area_at_risk_km2 = forest_at_risk.sum() * (resolution**2) / 1e6
print(f"Total forest area at risk of bamboo encroachment: {total_area_at_risk_km2:.2f} km²")

In [ ]:
from shapely.geometry import Point

# Ensure both layers are in the same CRS
bamboo_layer = bamboo_layer.to_crs(forest_layer.crs)

# Perform the union operation
bamboo_forest_union = gpd.overlay(bamboo_layer, forest_layer, how='union')

# Add a column to categorize the geometries
bamboo_forest_union['Category'] = bamboo_forest_union.apply(
    lambda row: "Bamboo and Forest" if row.geometry in bamboo_layer.geometry and row.geometry in forest_layer.geometry 
    else ("Bamboo" if row.geometry in bamboo_layer.geometry 
          else "Forest"),
    axis=1
)

# Define colors for the categories
color_map = {
    "Bamboo": "lightgreen",
    "Forest": "darkgreen",
    "Bamboo and Forest": "blue"
}

# Map colors to the GeoDataFrame
bamboo_forest_union['color'] = bamboo_forest_union['Category'].map(color_map)

# Plot the result
fig, ax = plt.subplots(figsize=(12, 10))
bamboo_forest_union.plot(ax=ax, color=bamboo_forest_union['color'], edgecolor='black', linewidth=0.1)

# Add legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label)
           for label, color in color_map.items()]
ax.legend(handles=handles, title="Categories", loc="upper left")

# Add title
plt.title("Bamboo and Forest Union Map", fontsize=16)

# Add scale bar and north arrow
def add_scale_bar(ax, length_km=20, location=(0.9, 0.05), linewidth=2, tick_height=0.01, label_offset=0.02, km_offset=0.03):
    x, y = location
    bar_half_length = 0.05

    ax.plot(
        [x - bar_half_length, x + bar_half_length], [y, y],
        transform=ax.transAxes, color='black', linewidth=linewidth
    )

    tick_positions = [x - bar_half_length, x, x + bar_half_length]
    for pos in tick_positions:
        ax.plot(
            [pos, pos], [y - tick_height / 2, y + tick_height / 2],
            transform=ax.transAxes, color='black', linewidth=linewidth
        )

    ax.text(
        x - bar_half_length, y - tick_height - label_offset, "0", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x, y - tick_height - label_offset, f"{length_km // 2}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x + bar_half_length, y - tick_height - label_offset, f"{length_km}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x + bar_half_length + km_offset, y, "km", transform=ax.transAxes, 
        ha='left', va='center', fontsize=12
    )

def add_north_arrow(ax, location=(0.9, 0.9), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate(
        '', xy=(x, y + size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', headwidth=10, headlength=15, width=5)
    )
    ax.text(
        x, y + size + label_offset, "N", transform=ax.transAxes,
        fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black"
    )

add_north_arrow(ax)
add_scale_bar(ax)

plt.tight_layout()
plt.show()

In [ ]:
# Perform an intersection between forest and bamboo layers
forest_bamboo_intersection = gpd.overlay(forest_layer, bamboo_layer, how='intersection')

# Calculate the area of overlapping regions
forest_bamboo_intersection['area_m2'] = forest_bamboo_intersection.geometry.area
forest_bamboo_intersection['area_km2'] = forest_bamboo_intersection['area_m2'] / 1e6

# Summarize the total area of forest affected by bamboo
total_area_affected_km2 = forest_bamboo_intersection['area_km2'].sum()
print(f"Total forest area at risk of bamboo encroachment: {total_area_affected_km2:.2f} km²")

In [ ]:
# Step 1: Filter Bamboo and Forest Classes
bamboo_layer = terrestrial_landcover[terrestrial_landcover['Classify'].isin(bamboo_categories)]
forest_layer = terrestrial_landcover[terrestrial_landcover['Classify'].isin(forest_categories)]

In [ ]:
# Step 2: Create a 500m Buffer Around Bamboo
bamboo_buffer = bamboo_layer.copy()
bamboo_buffer['geometry'] = bamboo_buffer.geometry.buffer(500)

In [ ]:
# Step 3: Intersect the Buffer with Forest Areas
forest_within_buffer = gpd.overlay(forest_layer, bamboo_buffer, how='intersection')

In [ ]:
forest_within_buffer.plot()

In [ ]:
# Step 4: Calculate Areas
# Calculate the total forest area
forest_layer['area_m2'] = forest_layer.geometry.area
total_forest_area = forest_layer['area_m2'].sum()

# Calculate the area of forest within the buffer
forest_within_buffer['area_m2'] = forest_within_buffer.geometry.area
forest_within_buffer_area = forest_within_buffer['area_m2'].sum()

# Step 5: Calculate the Proportion
proportion_within_buffer = (forest_within_buffer_area / total_forest_area) * 100

# Print Results
print(f"Total Forest Area: {total_forest_area / 1e6:.2f} km²")
print(f"Forest Area within 500m of Bamboo: {forest_within_buffer_area / 1e6:.2f} km²")
print(f"Proportion of Forest Area within 500m of Bamboo: {proportion_within_buffer:.2f}%")

In [ ]:
# Define the selected categories you want to display
selected_categories = ['Open dry forest: short', 'Open dry forest: tall',
                       'Disturbed broadleaved forest', 'Mixed land use: agriculture and bamboo', 
                       'Mixed land use: bamboo and forest', 'Closed broadleaved forest', 'Bamboo', 
                       'Secondary forest', 'Mixed land use: agriculture and bamboo', 
                       'Mixed land use: agriculture with secondary forest', 'Mixed land use: agriculture with forest or pine plantation']  # Add or modify categories as needed

# Filter the terrestrial_landcover GeoDataFrame
selected_landcover = terrestrial_landcover[terrestrial_landcover['Classify'].isin(selected_categories)]

# Display the filtered GeoDataFrame
display(selected_landcover.head())

In [ ]:
# Assign colors to the selected categories
category_colors = {
    'Open dry forest: short': '#A4C639', 
    'Open dry forest: tall': '#A4C639', 
    'Disturbed broadleaved forest': '#00FF7F',
    'Mixed land use: bamboo and forest': '#6B8E23',
    'Closed broadleaved forest': '#006400',
    'Bamboo': '#DEB887',  # Burlywood
    'Secondary forest': '#FFD700', # gold
    'Mixed land use: agriculture and bamboo': '#DC143C',  # Crimson
    'Mixed land use: agriculture with secondary forest': '#FA8072',  # Salmon
    'Mixed land use: agriculture with forest or pine plantation': '#A52A2A',  # Brown
     # Add or modify categories as needed

}

# Map colors to the GeoDataFrame
selected_landcover['color'] = selected_landcover['Classify'].map(category_colors)

# Plot the results
fig, ax = plt.subplots(figsize=(12, 10))

# Plot the filtered land cover
selected_landcover.plot(
    ax=ax,
    color=selected_landcover['color'],
    edgecolor='black',
    linewidth=0.5
)

# Add a legend
legend_handles = [
    plt.Line2D([0], [0], marker='o', color=color, label=category, markersize=10, linestyle='None')
    for category, color in category_colors.items()
]
ax.legend(handles=legend_handles, title="Selected Land Use Categories", loc='upper right')

# Add a title
plt.title("Selected Land Use Categories Map", fontsize=16)

plt.show()

In [ ]:
# Specify the forest types you want to include
specific_forest_types = [
    'Open dry forest: short', 
    'Open dry forest: tall', 
    'Disturbed broadleaved forest', 
    'Closed broadleaved forest', 
    'Secondary forest'
]

In [ ]:
print(selected_landcover['Classify'].unique())

In [ ]:
bamboo_layer = selected_landcover[selected_landcover['Classify'] == 'Mixed land use: bamboo and forest']
bamboo_layer.plot()

In [ ]:
# Filter the bamboo and forest layers
bamboo_layer = selected_landcover[selected_landcover['Classify'] == 'Bamboo']
forest_layer = selected_landcover[selected_landcover['Classify'].isin(specific_forest_types)]

In [ ]:
# Create a 100m buffer around bamboo areas
bamboo_buffer = bamboo_layer.copy()
bamboo_buffer['geometry'] = bamboo_buffer.geometry.buffer(500)

In [ ]:
# Find the intersection between the forest layer and the bamboo buffer
forest_bamboo_intersection = gpd.overlay(forest_layer, bamboo_buffer, how='intersection')

In [ ]:
# Calculate the area of the intersected polygons
forest_bamboo_intersection['area_m2'] = forest_bamboo_intersection.geometry.area

# Sum the areas
total_forest_area_within_500m = forest_bamboo_intersection['area_m2'].sum() / 1e6  # Convert to km²

# Display the result
print(f"Total forest area within 500m of bamboo: {total_forest_area_within_500m:.2f} km²")

In [ ]:
# Ensure the CRS of both layers is the same
bamboo_layer = bamboo_layer.to_crs(jamaica_metric_grid_crs)
forest_layer = forest_layer.to_crs(jamaica_metric_grid_crs)

# Perform a spatial join to find the nearest forest for each bamboo feature
nearest_forest = gpd.sjoin_nearest(
    bamboo_layer,
    forest_layer[['Classify', 'geometry']],  # Keep only 'Classify' and 'geometry' from forest_layer
    how='left',
    distance_col='distance_to_forest'
)

# Inspect the result to verify columns
print("Columns in nearest_forest after join:", nearest_forest.columns)

# Display the result
# Use 'Classify_right' because it's the forest layer's 'Classify' column
display(nearest_forest[['Classify_right', 'distance_to_forest']].rename(
    columns={'Classify_right': 'Forest Type', 'distance_to_forest': 'Distance to Forest (m)'}
))

# Convert distance to meters or kilometers (if CRS is metric)
nearest_forest['distance_to_forest_km'] = nearest_forest['distance_to_forest'] / 1000  # Convert to km

# Calculate summary statistics (optional)
min_distance = nearest_forest['distance_to_forest_km'].min()
max_distance = nearest_forest['distance_to_forest_km'].max()
mean_distance = nearest_forest['distance_to_forest_km'].mean()

print(f"Minimum distance to forest: {min_distance:.2f} km")
print(f"Maximum distance to forest: {max_distance:.2f} km")
print(f"Average distance to forest: {mean_distance:.2f} km")

In [ ]:
# Plot bamboo and forest layers with the nearest forest connections
fig, ax = plt.subplots(figsize=(12, 10))

# Plot bamboo areas
bamboo_layer.plot(ax=ax, color='green', label='Bamboo')

# Plot forest areas
forest_layer.plot(ax=ax, color='blue', alpha=0.5, label='Forest')

# Add lines showing the nearest connections and annotate forest types
for _, row in nearest_forest.iterrows():
    bamboo_geom = row['geometry']  # Bamboo geometry
    nearest_index = row['index_right']  # Index of the nearest forest geometry
    nearest_geom = forest_layer.loc[nearest_index, 'geometry']  # Nearest forest geometry
    forest_type = forest_layer.loc[nearest_index, 'Classify']  # Forest type

    # Create a line between the bamboo and the nearest forest
    line = gpd.GeoSeries([bamboo_geom, nearest_geom]).unary_union  # Create a line
    gpd.GeoSeries([line]).plot(ax=ax, color='red', linewidth=0.5)

    # Annotate the forest type near the forest point
    centroid = nearest_geom.centroid  # Get a point to place the annotation
    ax.text(centroid.x, centroid.y, forest_type, fontsize=8, color='black', ha='center', alpha=0.8)

# Add a custom legend
legend_handles = [
    mpatches.Patch(color='green', label='Bamboo'),
    mpatches.Patch(color='blue', label='Forest'),
    plt.Line2D([0], [0], color='red', linewidth=1, label='Nearest Connection')
]
ax.legend(handles=legend_handles, title="Legend", loc='upper right')

# Add a title
plt.title("Bamboo to Nearest Forest Connections with Forest Types", fontsize=16)

# Show the plot
plt.show()

In [ ]:
# Reset index to avoid alignment issues
nearest_forest = nearest_forest.reset_index(drop=True)
bamboo_layer = bamboo_layer.reset_index(drop=True)

# Categorize bamboo by distance to nearest forest
bamboo_layer['distance_category'] = pd.cut(
    nearest_forest['distance_to_forest'],  # Use the calculated distances
    bins=[0, 100, 500, 1000, np.inf],
    labels=['0-100m (Red)', '100-500m (Orange)', '500-1000m (Yellow)', '>1000m (Beige)']
)

# Assign colors to the bamboo distance categories
bamboo_distance_colors = {
    '0-100m (Red)': 'red',
    '100-500m (Orange)': 'orange',
    '500-1000m (Yellow)': 'yellow',
    '>1000m (Beige)': 'beige'
}

# Map the colors to the bamboo layer
bamboo_layer['color'] = bamboo_layer['distance_category'].map(bamboo_distance_colors)

# Prepare forest types for visualization
forest_layer['forest_type_color'] = forest_layer['Classify'].map({
    'Open dry forest: short': '#A4C639',
    'Open dry forest: tall': '#90EE90',
    'Disturbed broadleaved forest': '#228B22',
    'Closed broadleaved forest': '#006400',
    'Secondary forest': '#FFD700'
})

# Plot the bamboo and forest layers with the distance categories
fig, ax = plt.subplots(figsize=(15, 12))

# Plot bamboo areas categorized by distance
bamboo_layer.plot(
    ax=ax,
    color=bamboo_layer['color'],
    edgecolor='black',
    linewidth=0.5,
    label='Bamboo'
)

# Plot forest areas with forest type colors
forest_layer.plot(
    ax=ax,
    color=forest_layer['forest_type_color'],
    edgecolor='black',
    linewidth=0.5,
    alpha=0.7,
    label='Forest'
)

# Add legend for bamboo distance categories
bamboo_legend_handles = [
    mpatches.Patch(color=color, label=label)
    for label, color in bamboo_distance_colors.items()
]
ax.legend(
    handles=bamboo_legend_handles,
    title="Bamboo Distance to Nearest Forest",
    loc='upper left'
)

# Add legend for forest types
forest_legend_handles = [
    mpatches.Patch(color=color, label=forest_type)
    for forest_type, color in forest_layer['forest_type_color'].dropna().unique().items()
]
ax.legend(
    handles=forest_legend_handles,
    title="Forest Types",
    loc='upper right'
)

# Add a title
plt.title("Bamboo Proximity to Forests by Distance Category", fontsize=16)

plt.show()

In [ ]:
overlap = gpd.overlay(bamboo_layer, forest_layer, how='intersection')
if not overlap.empty:
    print(f"Number of overlapping features: {len(overlap)}")
    overlap.plot()
else:
    print("No overlapping bamboo and forest features.")

In [ ]:
farthest_bamboo = nearest_forest[nearest_forest['distance_to_forest_km'] == max_distance]
display(farthest_bamboo)